# Лекция 10. Объектная модель и качество кода

Класс нужен не потому, что программа стала большой, а когда данные и операции над ними образуют один объект с правилами. В этой лекции соберём небольшой отчёт по транзакциям и увидим, как классы, `dataclass`, композиция, протоколы и тестируемые зависимости решают вполне обычные задачи.

## Цели

После лекции вы сможете:

- выбирать между функцией, классом и `dataclass`;
- создавать экземпляры и сохранять инварианты;
- различать атрибуты экземпляра и класса;
- применять композицию, наследование и `Protocol` по назначению;
- понимать роль аннотаций типов;
- писать простой декоратор с `wraps`;
- проектировать явные зависимости, которые легко тестировать;
- распознавать типичные ошибки с общим изменяемым состоянием.

## Перед началом

Нужны функции, модель объектов, контейнеры и аннотации предыдущих занятий. На классы и состояние заложено около 30 минут, на `dataclass` — 20 минут, на композицию, наследование и протоколы — 25 минут, на декораторы, контринтуитивные примеры, самопроверку и вопросы — 15 минут.

## Функция или класс

Обычная функция остаётся лучшим выбором, если результат зависит только от явных аргументов и между вызовами не нужно сохранять состояние.

Класс полезен, когда:

- несколько операций работают с одним состоянием;
- состояние должно всегда удовлетворять правилам;
- объект передаётся между частями программы как единое целое;
- нужно подменять одну реализацию другой через общий интерфейс.

Не каждый словарь надо немедленно превращать в объект.

## Первый класс: счёт с инвариантом

У счёта есть идентификатор и баланс. Пополнение и списание должны проверять положительную сумму, а списание — доступный остаток. Если держать баланс отдельно от функций, любой код сможет случайно обойти эти правила.

In [ ]:
from decimal import Decimal

class Account:
    def __init__(self, account_id: str, balance: Decimal = Decimal("0")) -> None:
        if not account_id.strip():
            raise ValueError("account_id must not be empty")
        if balance < 0:
            raise ValueError("balance must not be negative")
        self.account_id = account_id
        self._balance = balance

    @property
    def balance(self) -> Decimal:
        return self._balance

    def deposit(self, amount: Decimal) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount

    def withdraw(self, amount: Decimal) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount

account = Account("a-1", Decimal("100"))
account.deposit(Decimal("25"))
account.withdraw(Decimal("40"))
assert account.balance == Decimal("85")

## Экземпляр, `self` и метод

`Account(...)` создаёт новый экземпляр и вызывает его `__init__`. Атрибуты `self.account_id` и `self._balance` принадлежат конкретному экземпляру.

`self` не является специальным ключевым словом, но это обязательное соглашение. Вызов `account.deposit(x)` концептуально эквивалентен `Account.deposit(account, x)`: Python связывает найденную в классе функцию с экземпляром.

In [ ]:
another = Account("a-2")
Account.deposit(another, Decimal("10"))
bound_deposit = another.deposit
bound_deposit(Decimal("5"))
assert another.balance == Decimal("15")
assert bound_deposit.__self__ is another

## Инвариант и атомарность изменения

Инвариант — условие, истинное после создания объекта и после каждого успешного публичного метода. Здесь баланс неотрицателен.

Проверки выполняются **до** изменения. Если `withdraw` поднимает исключение, баланс остаётся прежним. Это важнее самого синтаксиса класса: объект не должен застревать в наполовину изменённом состоянии.

In [ ]:
before = account.balance
try:
    account.withdraw(Decimal("1000"))
except ValueError:
    pass
assert account.balance == before

## Публичное и внутреннее

Один подчёркивающий префикс, как в `_balance`, означает: это внутренняя деталь, внешний код не должен на неё опираться. Python не делает атрибут недоступным, поэтому защита строится на понятном публичном API и соглашении.

`@property` позволяет читать `account.balance` как атрибут, но выполнять код метода. Setter здесь не определён: обычное присваивание `account.balance = ...` запрещено.

## Атрибут экземпляра и атрибут класса

Атрибут экземпляра создаётся через `self.name` и хранит индивидуальное состояние. Атрибут класса записан в теле класса и обычно задаёт общую константу или поведение.

Если изменяемый контейнер поместить в атрибут класса, он станет общим для всех экземпляров. Список операций счёта поэтому должен создаваться внутри `__init__`, а не один раз в теле класса.

In [ ]:
class LabeledAccount:
    currency = "RUB"  # общая неизменяемая настройка

    def __init__(self, account_id):
        self.account_id = account_id
        self.operations = []  # новый список для экземпляра

left = LabeledAccount("left")
right = LabeledAccount("right")
left.operations.append(100)
assert right.operations == []
assert left.currency == right.currency == "RUB"

## `dataclass`: класс-запись без служебного кода

Транзакция в основном хранит данные. Декоратор `@dataclass` по аннотированным полям создаёт `__init__`, понятный `__repr__` и сравнение значений. Методы и проверки при этом остаются обычными методами класса.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Transaction:
    transaction_id: str
    category: str
    amount: Decimal

    def __post_init__(self) -> None:
        if not self.transaction_id.strip():
            raise ValueError("transaction_id must not be empty")
        if not self.category.strip():
            raise ValueError("category must not be empty")
        if self.amount <= 0:
            raise ValueError("amount must be positive")

food = Transaction("t-1", "food", Decimal("120.50"))
assert food == Transaction("t-1", "food", Decimal("120.50"))
print(food)

## `__post_init__` и `frozen=True`

Сгенерированный `__init__` присваивает поля, затем вызывает `__post_init__`. Там удобно проверять связанные с полями правила.

`frozen=True` запрещает обычное переназначение полей после создания и делает намерение явным. Это не превращает весь граф вложенных объектов в глубокую неизменяемую структуру.

## Новый изменяемый контейнер через `default_factory`

Правило из лекции 1 действует и здесь: список нельзя делить между экземплярами. Для поля `dataclass` используют `field(default_factory=list)`. Фабрика вызывается при создании каждого объекта.

In [ ]:
from dataclasses import field

@dataclass
class ReportDraft:
    title: str
    notes: list[str] = field(default_factory=list)

first_draft = ReportDraft("January")
second_draft = ReportDraft("February")
first_draft.notes.append("check tax")
assert second_draft.notes == []

## Композиция: объект состоит из других объектов

Отчёт **содержит** транзакции, поэтому композиция естественнее наследования от `list`. Внутреннее хранение кортежем защищает набор записей от случайного изменения через исходный список.

In [ ]:
class BudgetReport:
    def __init__(self, transactions) -> None:
        self._transactions = tuple(transactions)

    def total(self) -> Decimal:
        return sum((item.amount for item in self._transactions), Decimal("0"))

    def total_by_category(self) -> dict[str, Decimal]:
        result = {}
        for item in self._transactions:
            result[item.category] = result.get(item.category, Decimal("0")) + item.amount
        return result

source = [food]
report = BudgetReport(source)
source.clear()
assert report.total() == Decimal("120.50")

## Наследование: разновидность с тем же контрактом

Наследование уместно, если объект подкласса действительно можно использовать там, где ожидается базовый. Международная комиссия является вариантом комиссии: она сохраняет метод `calculate(amount)` и правило положительного результата, но расширяет расчёт.

In [ ]:
class FeePolicy:
    def calculate(self, amount: Decimal) -> Decimal:
        if amount <= 0:
            raise ValueError("amount must be positive")
        return amount * Decimal("0.01")

class InternationalFeePolicy(FeePolicy):
    def calculate(self, amount: Decimal) -> Decimal:
        base_fee = super().calculate(amount)
        return base_fee + Decimal("50")

assert FeePolicy().calculate(Decimal("1000")) == Decimal("10.00")
assert InternationalFeePolicy().calculate(Decimal("1000")) == Decimal("60.00")

### Наследование — не способ получить чужие методы любой ценой

Если подкласс меняет смысл операции, усиливает предусловия или неожиданно ломает результат, клиент базового типа больше не может его подставить. Для отношения «использует» или «содержит» выбирайте композицию.

Сервис оплаты **использует** политику комиссии, поэтому получает её в конструкторе, а не наследуется от неё.

In [ ]:
class PaymentService:
    def __init__(self, fee_policy) -> None:
        self._fee_policy = fee_policy

    def total_to_charge(self, amount: Decimal) -> Decimal:
        return amount + self._fee_policy.calculate(amount)

service = PaymentService(InternationalFeePolicy())
assert service.total_to_charge(Decimal("1000")) == Decimal("1060.00")

## `Protocol`: контракт по форме, а не по родословной

Сервису нужен объект с методом `calculate(Decimal) -> Decimal`. Ему необязательно знать общий базовый класс. `Protocol` описывает структурный интерфейс для анализатора типов: любой подходящий объект совместим без явного наследования.

In [ ]:
from typing import Protocol

class CalculatesFee(Protocol):
    def calculate(self, amount: Decimal) -> Decimal:
        ...

class NoFee:
    def calculate(self, amount: Decimal) -> Decimal:
        return Decimal("0")

class TypedPaymentService:
    def __init__(self, fee_policy: CalculatesFee) -> None:
        self._fee_policy = fee_policy

    def total_to_charge(self, amount: Decimal) -> Decimal:
        return amount + self._fee_policy.calculate(amount)

assert TypedPaymentService(NoFee()).total_to_charge(Decimal("100")) == Decimal("100")

## Что делают аннотации

Аннотации документируют контракт и помогают статическому анализатору, IDE и читателю. Обычный интерпретатор не проверяет автоматически, что передан `CalculatesFee` или `Decimal`.

Проверку внешних данных всё равно выполняют на границе приложения. Внутри программы аннотации уменьшают число случайных несовместимостей, но не заменяют тесты и обработку ошибок.

## Явная зависимость создаёт точку подмены

Если `PaymentService` сам внутри метода создаёт HTTP-клиент или читает глобальную настройку, тест становится зависим от сети и окружения. Передача политики в конструктор делает зависимость видимой и позволяет использовать маленькую реализацию `NoFee`.

Это и есть простая dependency injection: объект получает нужную ему зависимость снаружи. Фреймворк для этого не требуется.

## Декоратор: функция получает функцию

Декоратор вызывается при выполнении `def` и возвращает объект, который будет связан с именем функции. Обычно это обёртка: она принимает `*args` и `**kwargs`, выполняет дополнительное действие и вызывает исходную функцию.

Для прикладного примера запишем имя вызванной операции в журнал аудита.

In [ ]:
from functools import wraps

audit_log = []

def audit_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        audit_log.append(func.__name__)
        return func(*args, **kwargs)
    return wrapper

@audit_call
def net_amount(amount: int, fee: int = 0) -> int:
    """Return amount after fee."""
    return amount - fee

assert net_amount(100, fee=7) == 93
assert audit_log == ["net_amount"]
assert net_amount.__name__ == "net_amount"

## Зачем `functools.wraps`

Без `@wraps(func)` имя, документация и часть служебных атрибутов указывали бы на `wrapper`. Это мешает справке, логам, тестовым инструментам и отладке.

Декоратор не должен без необходимости менять результат или подавлять исключения исходной функции. Дополнительное поведение обязано быть узким и очевидным.

## Практические признаки качественной модели

- У класса одна понятная ответственность и короткий публичный API.
- Конструктор создаёт корректный объект или поднимает понятную ошибку.
- Метод не оставляет состояние наполовину изменённым.
- Зависимости видны в параметрах или конструкторе.
- Композиция используется для отношения «содержит» и «использует».
- Наследование сохраняет поведение базового контракта.
- Протокол содержит только операции, реально нужные клиенту.
- Ввод-вывод отделён от предметных вычислений.
- Логи описывают события, а исключения сообщают об ошибках вызывающему коду.

## Как это раскладывается в проекте

```text
src/project_name/
├── models.py       # dataclass и предметные значения
├── services.py     # сценарии и композиция зависимостей
├── protocols.py    # узкие интерфейсы, если они переиспользуются
├── repositories.py # файлы, API, БД
└── cli.py          # внешний интерфейс

tests/
├── test_models.py
└── test_services.py
```

Это не обязательная раскладка для трёх функций. Модули разделяют тогда, когда у кода появились разные причины изменения.

## Неожиданно, но по правилам

Эти эффекты следуют из уже знакомой модели объектов: класс тоже является объектом, атрибуты могут быть общими, а декоратор выполняется при создании функции.

### 1. Изменяемый атрибут класса общий для экземпляров

Список создаётся один раз при выполнении тела класса. Оба экземпляра находят один объект через класс — тот же эффект, что у `items=[]` в параметрах функции.

In [ ]:
class BrokenDraft:
    notes = []

left = BrokenDraft()
right = BrokenDraft()
left.notes.append("shared")
assert right.notes == ["shared"]
assert left.notes is right.notes

### 2. Метод можно сохранить и вызвать позже

Выражение `account.deposit` создаёт связанный метод, который уже помнит экземпляр. Поэтому при последующем вызове передаётся только сумма.

In [ ]:
saved_method = account.deposit
saved_method(Decimal("5"))
assert account.balance == Decimal("90")
assert saved_method.__self__ is account

### 3. `frozen=True` не замораживает вложенный список

Декоратор запрещает переназначить поле, но не меняет тип объекта внутри него. Для глубокой неизменяемости выбирают неизменяемые поля, например кортеж.

In [ ]:
@dataclass(frozen=True)
class FrozenEnvelope:
    tags: list[str]

envelope = FrozenEnvelope(["new"])
envelope.tags.append("checked")
assert envelope.tags == ["new", "checked"]

### 4. Декоратор вызывается при `def`, обёртка — при вызове

До первого вызова декорированной функции уже успевает выполниться внешний декоратор. Это объясняет регистрацию маршрутов веб-фреймворка через `@app.get(...)`: маршрут связывается при импорте модуля.

In [ ]:
events = []

def register(func):
    events.append(("decorated", func.__name__))
    return func

@register
def build_report():
    events.append(("called", "build_report"))

assert events == [("decorated", "build_report")]
build_report()
assert events[-1] == ("called", "build_report")

### 5. Совместимость с `Protocol` не требует наследования

`NoFee` не упоминает `CalculatesFee`, но статически совместим по нужному методу. Это структурная типизация. Обычный `Protocol` при этом не предназначен для `isinstance` — его главная работа происходит в анализаторе типов.

## Самопроверка

1. Когда функция лучше класса?
2. Что делает `self` при вызове метода?
3. Чем атрибут экземпляра отличается от атрибута класса?
4. Где проверять инварианты создаваемого объекта?
5. Что генерирует `@dataclass`?
6. Зачем нужен `default_factory`?
7. В чём разница отношений «является» и «использует»?
8. Почему `Protocol` не требует наследования?
9. Что аннотации не проверяют автоматически?
10. Когда выполняется декоратор и зачем нужен `wraps`?
11. Как явная зависимость упрощает тест?

## Источники

- [Classes — Python tutorial](https://docs.python.org/3/tutorial/classes.html) — экземпляры, методы, атрибуты и наследование.
- [`dataclasses`](https://docs.python.org/3/library/dataclasses.html) — генерируемые методы, `field`, `frozen` и `__post_init__`.
- [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol) — структурные интерфейсы.
- [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps) — сохранение метаданных декорируемой функции.

## Итоги

- Класс объединяет состояние и операции, если вместе они поддерживают правила объекта.
- Атрибуты экземпляра индивидуальны; изменяемые атрибуты класса разделяются.
- `dataclass` убирает служебный код класса-записи, но не проектирует инварианты за нас.
- Композиция выражает «содержит» и «использует», наследование — настоящую разновидность с сохранённым контрактом.
- `Protocol` описывает узкую структурную зависимость без общей иерархии.
- Аннотации помогают до запуска, а проверки внешних данных работают во время выполнения.
- Декоратор оборачивает функцию; `wraps` сохраняет её метаданные.
- Явные зависимости, небольшие интерфейсы и атомарные изменения делают код тестируемым и предсказуемым.